# Proposed Framework Walkthrough

This notebook is a compact walkthrough of the proposed online bin-packing framework. It connects the main project components in the same order they are used during validation:

- instantiate the Gymnasium packing environment
- build the policy network and load checkpoint weights when available
- roll out the packing policy until the bin reaches a blocked state or target utilization
- visualize the packing sequence with an interactive replay
- run MCTS-based safe rearrangement from the blocked state
- optionally optimize and replay the rearrangement sequence

![Proposed method](../figures/Proposed_method.png)


## 1. Imports And Config

The demo uses `configs/test_demo.yaml` for reproducible defaults. The default agent is the checkpoint-backed policy. If `CHECKPOINT_PATH` does not exist, the notebook falls back to the deterministic greedy validation agent so it can still run on a fresh clone.


In [1]:
from copy import deepcopy
from dataclasses import replace
import importlib
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import gymnasium as gym
from IPython.display import IFrame, Markdown, display
from gymnasium.envs.registration import register

import packing.a_star as a_star_module
import packing.mcts as mcts_module
import packing.plans as plans_module
import packing.test_utils as test_utils_module
import packing.visualizer as visualizer_module

importlib.reload(plans_module)
importlib.reload(mcts_module)
importlib.reload(a_star_module)
importlib.reload(visualizer_module)
importlib.reload(test_utils_module)

from packing.agents import GreedyValidationAgent, PackingAgent
from packing.interactive_replay import InteractiveReplayRecorder
Visualizer = visualizer_module.Visualizer
execution_plan_from_mcts_plan = plans_module.execution_plan_from_mcts_plan
optimize_execution_plan = a_star_module.optimize_execution_plan
mcts = mcts_module.mcts
from packing.debug_utils import print_execution_steps, print_mcts_stats, print_mcts_steps
from packing.policy_loader import build_net, load_policy_weights, resolve_runtime_device, set_eval_seed
load_test_config = test_utils_module.load_test_config
record_pack_step = test_utils_module.record_pack_step
from packing_env.visualization import PackVisualizer


In [2]:
CONFIG_PATH = PROJECT_ROOT / "configs/test_demo.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / "train_outputs/random/policy_step.pth"
REPLAY_DIR = PROJECT_ROOT / "_plotly_live/notebook_demo"
SEED = 0

config = load_test_config(str(CONFIG_PATH))
config = replace(config, agent="policy")
config

def jupyter_file_url(path: Path) -> str:
    rel_path = path.resolve().relative_to(PROJECT_ROOT.resolve())
    return "/files/" + rel_path.as_posix()



TestConfig(ds_name='random', agent='policy', checkpoint='train_outputs/random/policy_step.pth', device=None, seed=0, num_sequences=1, iterations=100, max_unpack=6, target_util=0.8, max_steps=100, container_size=(600, 600, 600), buffer_space=0, remove_inscribed_ems=True, visualize=False, visual_dir='_plotly_live/mcts', visual_z_max=610.0, visual_port=8766, visual_delay_sec=0.5, visual_bind_host='127.0.0.1', visual_public_host='127.0.0.1', visual_poll_ms=300, hold_visual=False, save_replay=True, replay_path='_plotly_live/demo_compose', replay_interval_ms=700, mcts_max_child=3, optimize_sequence=False)

## 2. Instantiate The Gym Environment

The environment can be created directly through Gymnasium. It contains the container, buffer, EMS generator, stability validator, and height map.


In [3]:
if "OnlinePack-v2" not in gym.envs.registry:
    register(id="OnlinePack-v2", entry_point="packing_env:PackingEnv")

set_eval_seed(SEED)
env = gym.make(
    "OnlinePack-v2",
    k_placement=80,
    ds_name=config.ds_name,
    buffer_capacity=12,
    container_size=config.container_size,
    item_buffer_space=config.buffer_space,
    remove_inscribed_ems=config.remove_inscribed_ems,
)
# obs, info = env.reset(seed=SEED)

# print(type(env.unwrapped).__name__)
# print("initial buffer dims:", env.unwrapped.buffer.dims())
# print("initial utilization:", env.unwrapped.container.utilization)


## 3. Build Network And Load Weights

`PackingAgent` is the default policy interface. It wraps the actor and critic networks and exposes a small `predict(obs)` method. The cells below also show the lower-level network construction and checkpoint-loading call explicitly.


In [4]:
device = resolve_runtime_device(config.device)
actor, critic = build_net(device=str(device))

if CHECKPOINT_PATH.exists():
    load_policy_weights(actor, critic, str(CHECKPOINT_PATH), device)
    agent = PackingAgent(
        device=str(device),
        k_placement=env.unwrapped.k_placement,
        checkpoint_path=str(CHECKPOINT_PATH),
    )
    policy_name = f"policy checkpoint: {CHECKPOINT_PATH}"
else:
    actor.eval()
    critic.eval()
    agent = GreedyValidationAgent()
    policy_name = "greedy fallback, checkpoint not found"

print("device:", device)
print("actor:", actor.__class__.__name__)
print("critic:", critic.__class__.__name__)
print("packing policy:", policy_name)


device: cuda
actor: Actor
critic: Critic
packing policy: policy checkpoint: /home/gao/CJ_block_dockerize/train_outputs/random/policy_step.pth


## 4. Policy Packing Rollout

This section mirrors `test.py`'s `pack_until_blocked` workflow, but it is split into smaller notebook cells:

1. reset the environment and initialize replay/log containers
2. define one rollout helper that performs the repeated policy steps
3. run the helper and inspect the resulting pack log

The loop stops when the incoming item is not placeable, when target utilization is reached, or when `config.max_steps` is reached as a safety cap.


In [5]:
set_eval_seed(SEED)
env.reset(seed=SEED)
packing_env = env.unwrapped
initial_buffer = packing_env.buffer.dims()

print("initial buffer length:", len(initial_buffer))
print("initial item:", initial_buffer[0])
print("target utilization:", config.target_util)
print("max policy steps:", config.max_steps)


initial buffer length: 12
initial item: (240, 120, 300)
target utilization: 0.8
max policy steps: 100


### 4.1 Rollout Helper

The helper below keeps the policy loop close to `test.py`: sample the next buffer item, build pack data, call `agent.predict(obs)`, pack the selected box, update the buffer and other related stuff.


In [6]:
def run_policy_until_blocked(packing_env, agent, config, seed):
    pack_history = []
    pack_log = []
    replay_frames = [("Initial State", deepcopy(packing_env))]

    def result(blocked_item=None, blocked_env=None, target_reached=False):
        return {
            "pack_history": pack_history,
            "pack_log": pack_log,
            "replay_frames": replay_frames,
            "blocked_item": blocked_item,
            "blocked_env": blocked_env,
            "target_reached": target_reached,
        }

    for step in range(config.max_steps):
        item = packing_env.buffer.sample_item()
        obs = packing_env.get_pack_data(item)
        if not obs["placable"]:
            blocked_env = deepcopy(packing_env)
            title = f"Blocked Before MCTS at Pack Step {step} - seed {seed}"
            replay_frames.append((title, deepcopy(packing_env)))
            print(
                "blocked at step={}, utilization={:.4f}, placed={}, unpackable={}".format(
                    step,
                    packing_env.container.utilization,
                    len(packing_env.container.placed_items),
                    len(packing_env.container.unpackable_boxes),
                )
            )
            return result(blocked_item=item, blocked_env=blocked_env)

        box, (_, action_idx, predicted_value) = agent.predict(obs)
        selected_ems = packing_env.ems_list[action_idx % packing_env.k_placement]
        packing_env.pack(box, selected_ems=selected_ems)
        packing_env.buffer.update(item)
        packing_env.validate_packing_state()

        pack_history.append(record_pack_step(box, packing_env))
        title = f"Pack Step {step + 1} - seed {seed}"
        replay_frames.append((title, deepcopy(packing_env)))
        pack_log.append(
            {
                "step": step + 1,
                "action_idx": action_idx,
                "item_dim": item.raw().astype(int).tolist(),
                "box": repr(box),
                "predicted_value": float(predicted_value),
                "utilization": packing_env.container.utilization,
            }
        )
        print(
            "{} step={}, utilization={:.4f}, placed={}".format(
                config.agent,
                step,
                packing_env.container.utilization,
                len(packing_env.container.placed_items),
            )
        )

        if packing_env.container.utilization >= config.target_util:
            print("target utilization reached before MCTS was needed")
            return result(target_reached=True)

    raise AssertionError(f"no blocked item found within {config.max_steps} steps")


### 4.2 Run The Policy

After this cell runs, later sections reuse `pack_history`, `replay_frames`, `blocked_item`, and `blocked_env` for replay and MCTS.


In [7]:
rollout = run_policy_until_blocked(packing_env, agent, config, SEED)

pack_history = rollout["pack_history"]
pack_log = rollout["pack_log"]
replay_frames = rollout["replay_frames"]
blocked_item = rollout["blocked_item"]
blocked_env = rollout["blocked_env"]
target_reached = rollout["target_reached"]

print("placed items:", len(packing_env.container.placed_items))
print("replay frames:", len(replay_frames))
print("target reached:", target_reached)
print("blocked item:", None if blocked_item is None else blocked_item.raw().astype(int).tolist())


policy step=0, utilization=0.0400, placed=1
policy step=1, utilization=0.1000, placed=2
policy step=2, utilization=0.1480, placed=3
policy step=3, utilization=0.1800, placed=4
policy step=4, utilization=0.2400, placed=5
policy step=5, utilization=0.2880, placed=6
policy step=6, utilization=0.3180, placed=7
policy step=7, utilization=0.3780, placed=8
policy step=8, utilization=0.4530, placed=9
policy step=9, utilization=0.4770, placed=10
policy step=10, utilization=0.5170, placed=11
policy step=11, utilization=0.5410, placed=12
policy step=12, utilization=0.5530, placed=13
policy step=13, utilization=0.5710, placed=14
policy step=14, utilization=0.6710, placed=15
policy step=15, utilization=0.7210, placed=16
policy step=16, utilization=0.7510, placed=17
blocked at step=17, utilization=0.7510, placed=17, unpackable=7
placed items: 17
replay frames: 19
target reached: False
blocked item: [240, 240, 120]


In [8]:
lines = [
    "| step | item dim | action idx | predicted value | utilization |",
    "|---:|---|---:|---:|---:|",
]
for row in pack_log:
    lines.append(
        f"| {row['step']} | `{row['item_dim']}` | {row['action_idx']} | "
        f"{row['predicted_value']:.4f} | {row['utilization']:.4f} |"
    )
display(Markdown("\n".join(lines)))


| step | item dim | action idx | predicted value | utilization |
|---:|---|---:|---:|---:|
| 1 | `[240, 120, 300]` | 0 | 0.7216 | 0.0400 |
| 2 | `[240, 300, 180]` | 81 | 0.6835 | 0.1000 |
| 3 | `[240, 180, 240]` | 4 | 0.6254 | 0.1480 |
| 4 | `[240, 120, 240]` | 5 | 0.5727 | 0.1800 |
| 5 | `[180, 240, 300]` | 2 | 0.5480 | 0.2400 |
| 6 | `[240, 240, 180]` | 1 | 0.4654 | 0.2880 |
| 7 | `[180, 300, 120]` | 84 | 0.3927 | 0.3180 |
| 8 | `[300, 240, 180]` | 8 | 0.3819 | 0.3780 |
| 9 | `[300, 300, 180]` | 1 | 0.3286 | 0.4530 |
| 10 | `[180, 240, 120]` | 84 | 0.2635 | 0.4770 |
| 11 | `[300, 120, 240]` | 5 | 0.2313 | 0.5170 |
| 12 | `[240, 120, 180]` | 86 | 0.1967 | 0.5410 |
| 13 | `[120, 180, 120]` | 87 | 0.1729 | 0.5530 |
| 14 | `[120, 180, 180]` | 1 | 0.1597 | 0.5710 |
| 15 | `[300, 300, 240]` | 4 | 0.1460 | 0.6710 |
| 16 | `[300, 120, 300]` | 3 | 0.0537 | 0.7210 |
| 17 | `[120, 180, 300]` | 3 | 0.0397 | 0.7510 |

## 5. Interactive Policy Replay

The replay below records every packing step as a Plotly frame. Use **Prev**, **Next**, **Play**, or the slider to inspect each packed item. You can still rotate each 3D plot while moving through the frames.


In [9]:
replay_path = REPLAY_DIR / "packing_steps.html"
recorder = InteractiveReplayRecorder(str(replay_path), interval_ms=700)

replay_visualizer = PackVisualizer(
    replay_frames[0][1],
    title="Notebook Packing Replay",
    show_anchor=True,
    show_ems=False,
)

for frame_idx, (title, snapshot) in enumerate(replay_frames):
    replay_visualizer.env = snapshot
    replay_visualizer.title = title
    if frame_idx == 0:
        replay_visualizer.reset_history()
    fig, _ = replay_visualizer.refresh()
    recorder.capture(title, fig)

recorder.save()
display(IFrame(src=jupyter_file_url(replay_path), width="100%", height=760))


Saved interactive replay: /home/gao/CJ_block_dockerize/_plotly_live/notebook_demo/packing_steps.html (19 frames)


## 6. Run MCTS From The Blocked State

This mirrors the MCTS part of `test.py`: use the true blocked item and blocked environment produced by the packing loop, run `mcts(...)`, convert the result to an execution plan, and record replay snapshots if a sequence is found.


In [10]:
mcts_result = None
mcts_stats = None
execution_plan = None
mcts_replay_path = REPLAY_DIR / "mcts_replay.html"

if blocked_item is None:
    print("MCTS skipped because no blocked item was found.")
else:
    mcts_result, mcts_stats = mcts(
        deepcopy(blocked_env),
        agent,
        new_item=blocked_item,
        iterations=config.iterations,
        max_unpack=config.max_unpack,
        Uti_requirement=config.target_util,
        return_info=True,
        max_child=config.mcts_max_child,
    )
    print_mcts_stats(mcts_stats)

    if mcts_result is None:
        print("MCTS did not find an unpack/repack sequence for this blocked state.")
    else:
        print_mcts_steps(mcts_result)
        execution_plan = execution_plan_from_mcts_plan(mcts_result)
        replay_config = replace(
            config,
            save_replay=True,
            replay_path=str(mcts_replay_path),
        )
        replay_helper = Visualizer(replay_config)
        replay_helper.save_sequence_replay(
            seed=SEED,
            initial_buffer=initial_buffer,
            pack_history=pack_history,
            mcts_used=True,
            execution_plan=execution_plan,
            step_prefix="MCTS Step",
            start_from_blocked=True,
        )
        print("MCTS replay saved to:", mcts_replay_path)



MCTS search stats:
  iterations requested: 100
  iterations run: 2
  rollouts: 2
  expanded nodes: 2
  visited states: 3
  max children per node: 3
  root branches: 2
  success iteration: 2
  best unpack depth: 1
  best reward: 0.788270345300436
  timing breakdown:
    select: 0.0000s total, 0.0000s/rollout
    expand: 0.1003s total, 0.0501s/rollout
    child deepcopy: 0.0522s total, 0.0261s/rollout
    child unpack: 0.0480s total, 0.0240s/rollout
    rollout deepcopy: 0.0518s total, 0.0259s/rollout
    rollout: 0.0548s total, 0.0274s/rollout
    backprop: 0.0000s total, 0.0000s/rollout

MCTS final action sequence:
  unpack 1: Item(pos=(0.0,360.0,480.0), dim=(180,120,120))
  pack 1: incoming item -> Item(pos=(0.0,360.0,480.0), dim=(240,240,120))
  pack 2: holding item Item(pos=(0.0,360.0,480.0), dim=(180,120,120)) -> Item(pos=(240.0,420.0,480.0), dim=(120,180,120))
Saved interactive replay: /home/gao/CJ_block_dockerize/_plotly_live/notebook_demo/mcts_replay.html (4 frames)
MCTS replay

## 7. Interactive MCTS Replay

If MCTS found a plan, this replay starts at the blocked state and then shows each unpack and pack operation. Use the controls to inspect how the rearrangement creates space for the incoming item.


In [11]:
if execution_plan is None:
    print("No MCTS replay to show.")
elif not mcts_replay_path.exists():
    print(f"Replay file was not found: {mcts_replay_path}")
else:
    display(IFrame(src=jupyter_file_url(mcts_replay_path), width="100%", height=760))


## 8. Optimize Operation Sequence

The raw MCTS plan lists unpack operations first and pack operations second. The optimizer tries to shorten that into an executable sequence, for example by replacing an unpack-then-pack pair with one direct repack operation. The optimized sequence is saved as a separate interactive replay.


In [12]:
optimized_execution_plan = None
optimized_replay_path = REPLAY_DIR / "mcts_optimized_replay.html"

if mcts_result is None or blocked_env is None:
    print("No MCTS result to optimize.")
else:
    optimized_execution_plan = optimize_execution_plan(blocked_env, mcts_result)
    print_execution_steps(optimized_execution_plan)

    optimized_replay_config = replace(
        config,
        save_replay=True,
        replay_path=str(optimized_replay_path),
    )
    optimized_replay_helper = Visualizer(optimized_replay_config)
    optimized_replay_helper.save_sequence_replay(
        seed=SEED,
        initial_buffer=initial_buffer,
        pack_history=pack_history,
        mcts_used=True,
        execution_plan=optimized_execution_plan,
        step_prefix="Optimized Step",
        start_from_blocked=True,
    )
    print("Optimized replay saved to:", optimized_replay_path)



Optimized execution sequence:
  1: repack Item(pos=(0.0,360.0,480.0), dim=(180,120,120)) -> Item(pos=(240.0,420.0,480.0), dim=(120,180,120))
  2: pack incoming item -> Item(pos=(0.0,360.0,480.0), dim=(240,240,120))
Saved interactive replay: /home/gao/CJ_block_dockerize/_plotly_live/notebook_demo/mcts_optimized_replay.html (3 frames)
Optimized replay saved to: /home/gao/CJ_block_dockerize/_plotly_live/notebook_demo/mcts_optimized_replay.html


## 9. Interactive Optimized Replay

This replay starts at the blocked state and shows the optimized execution order. It should usually have the same final packed state as the raw MCTS replay, but fewer or clearer operations.


In [13]:
if optimized_execution_plan is None:
    print("No optimized replay to show.")
elif not optimized_replay_path.exists():
    print(f"Replay file was not found: {optimized_replay_path}")
else:
    display(IFrame(src=jupyter_file_url(optimized_replay_path), width="100%", height=760))


## 10. Command-Line Smoke Demo

For the automated smoke workflow, use the script-level demo config:

```bash
python test.py --config configs/test_demo.yaml --replay-path _plotly_live/demo
```

The notebook above is more explicit because it shows the environment, network, policy, packing loop, MCTS search, and visualization objects separately.
